# CoverAI — Model Training Walkthrough

Interactive notebook for training and evaluating the XGBoost recommendation model.
- Section 1: Generate/load data
- Section 2: Feature preprocessing
- Section 3: Train XGBoost with cross-validation
- Section 4: Evaluate — AUC, Precision@K, NDCG@K
- Section 5: SHAP feature importance
- Section 6: A/B comparison vs rule-based baseline

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, RocCurveDisplay, PrecisionRecallDisplay

plt.style.use('dark_background')
print(f'XGBoost version: {xgb.__version__}')

## 1. Data Preparation

In [ ]:
from ml.training.data_generator import generate_interaction_dataset
from ml.training.preprocessor import RecommendationPreprocessor

df = generate_interaction_dataset(n_users=5000)

pp = RecommendationPreprocessor()
X = pp.fit_transform(df)
y = df['label'].values

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {X_train.shape} | Val: {X_val.shape}')
print(f'Positive rate — Train: {y_train.mean():.2%} | Val: {y_val.mean():.2%}')

## 2. Train XGBoost

In [ ]:
model = xgb.XGBClassifier(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    gamma=0.1,
    scale_pos_weight=3.0,
    tree_method='hist',
    eval_metric='auc',
    early_stopping_rounds=30,
    random_state=42,
    use_label_encoder=False,
    verbosity=0,
)

model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=50,
)
print(f'Best iteration: {model.best_iteration}')

## 3. Evaluation

In [ ]:
from ml.evaluation.evaluator import RecommendationEvaluator
from sklearn.metrics import classification_report

y_prob = model.predict_proba(X_val)[:, 1]
y_pred = (y_prob >= 0.5).astype(int)

print(f'AUC-ROC: {roc_auc_score(y_val, y_prob):.4f}')
print()
print(classification_report(y_val, y_pred, target_names=['No Buy', 'Buy']))

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

RocCurveDisplay.from_predictions(y_val, y_prob, ax=ax1, color='#7c3aed',
    name=f'XGBoost (AUC={roc_auc_score(y_val, y_prob):.3f})')
ax1.plot([0,1],[0,1], 'k--', alpha=0.4)
ax1.set_title('ROC Curve')

PrecisionRecallDisplay.from_predictions(y_val, y_prob, ax=ax2, color='#38bdf8',
    name='XGBoost')
ax2.set_title('Precision-Recall Curve')

plt.tight_layout()
plt.savefig('../data/processed/model_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Feature Importance (Gain)

In [ ]:
from ml.features.feature_definitions import get_feature_names

feat_names = get_feature_names()
importances = model.feature_importances_
fi = pd.Series(importances, index=feat_names).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
fi.head(20).plot(kind='barh', ax=ax, color='#7c3aed', alpha=0.85)
ax.set_title('Top 20 Features by XGBoost Gain Importance', fontweight='bold')
ax.set_xlabel('Feature Importance (Gain)')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('../data/processed/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('Top 10 most important features:')
for i, (feat, imp) in enumerate(fi.head(10).items(), 1):
    bar = '█' * int(imp / fi.max() * 30)
    print(f'{i:2}. {feat:<35} {imp:.4f} {bar}')

## 5. SHAP Values

In [ ]:
try:
    import shap
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_val[:300])
    
    shap.summary_plot(
        shap_values, X_val[:300],
        feature_names=feat_names,
        max_display=15,
        show=False,
        plot_type='bar',
    )
    plt.title('SHAP Feature Importance (Mean |SHAP value|)', fontweight='bold')
    plt.tight_layout()
    plt.savefig('../data/processed/shap_importance.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('SHAP analysis complete')
except ImportError:
    print('SHAP not installed. Run: pip install shap')

## 6. A/B Comparison: ML vs Rule-Based

In [ ]:
from ml.evaluation.evaluator import run_offline_evaluation

# Temporarily save the trained model so the registry can load it
import joblib
from config.settings import get_settings
settings = get_settings()

settings.MODEL_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(model, settings.model_path)
pp.save(settings.scaler_path, settings.feature_names_path)

from ml.models.model_registry import model_registry
model_registry.load(force=True)

print('Running A/B evaluation (this takes ~30s)…')
results = run_offline_evaluation()

print('\n--- A/B Lift (ML vs Rule-Based) ---')
for k, v in results['lifts'].items():
    sign = '+' if v > 0 else ''
    print(f'  {k:<35}: {sign}{v:.2f}%')